# Macro Pipeline: WDI Download + Panel Build

Fetches WDI indicators, caches raw data, updates registry, and builds `macro_panel.parquet` with coverage reports.

In [1]:
from __future__ import annotations
from pathlib import Path
import io
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import requests
import hashlib
import yaml
from datetime import datetime
import wbgapi as wb

pd.options.mode.copy_on_write = True


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for p in [start] + list(start.parents):
        if (p / 'data').exists() and (p / 'notebooks').exists():
            return p
    raise ValueError('Repo root not found')

ROOT = find_repo_root()

PATHS = {
    'data_processed': ROOT / 'data' / 'processed',
    'data_raw': ROOT / 'data' / 'raw',
    'reports': ROOT / 'reports',
}


def read_parquet_pyarrow(path: str | Path) -> pd.DataFrame:
    table = pq.read_table(str(path))
    return table.to_pandas()


def write_parquet_pyarrow(df: pd.DataFrame, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, str(path))


REGISTRY_PATH = PATHS['data_raw'] / 'macro' / '_registry.yml'

def read_registry() -> list[dict]:
    if not REGISTRY_PATH.exists():
        return []
    with open(REGISTRY_PATH, 'r') as f:
        data = yaml.safe_load(f) or []
    return data


def write_registry(entries: list[dict]) -> None:
    REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(REGISTRY_PATH, 'w') as f:
        yaml.safe_dump(entries, f, sort_keys=False)


def update_registry(entry: dict) -> None:
    entries = read_registry()
    entries.append(entry)
    write_registry(entries)


AGGREGATES = set(wb.economy.aggregates())


In [2]:
# WDI indicator mapping
INDICATORS = {
    # Real activity / development
    'gdp_pc_real': 'NY.GDP.PCAP.KD',
    'gdp_growth': 'NY.GDP.MKTP.KD.ZG',
    'gdp_pc_growth': 'NY.GDP.PCAP.KD.ZG',
    'inv_gdp': 'NE.GDI.FTOT.ZS',
    'pop': 'SP.POP.TOTL',
    'pop_growth': 'SP.POP.GROW',

    # Prices / monetary
    'inflation_cpi': 'FP.CPI.TOTL.ZG',
    'exrate_lcu_per_usd': 'PA.NUS.FCRF',
    'credit_private_gdp': 'FS.AST.PRVT.GD.ZS',
    'money_broad_gdp': 'FM.LBL.BMNY.GD.ZS',

    # Fiscal / state size
    'gov_cons_gdp': 'NE.CON.GOVT.ZS',
    'tax_rev_gdp': 'GC.TAX.TOTL.GD.ZS',
    'debt_gdp': 'GC.DOD.TOTL.GD.ZS',

    # External sector
    'trade_gdp': 'NE.TRD.GNFS.ZS',
    'exports_gdp': 'NE.EXP.GNFS.ZS',
    'imports_gdp': 'NE.IMP.GNFS.ZS',
    'ca_gdp': 'BN.CAB.XOKA.GD.ZS',
    'natres_rents_gdp': 'NY.GDP.TOTL.RT.ZS',

    # Demography
    'urban_share': 'SP.URB.TOTL.IN.ZS',
    'dep_ratio': 'SP.POP.DPND',
}


In [3]:
# Fetch WDI with caching

def _cache_key(indicators: dict, start: int, end: int) -> str:
    payload = str(sorted(indicators.items())) + f"|{start}|{end}"
    return hashlib.sha1(payload.encode('utf-8')).hexdigest()[:12]


def fetch_wdi(indicators: dict[str,str], start: int, end: int) -> pd.DataFrame:
    key = _cache_key(indicators, start, end)
    cache_path = PATHS['data_raw'] / 'macro' / 'wdi' / f'cache_{key}.parquet'
    if cache_path.exists():
        print('Using cache:', cache_path)
        return read_parquet_pyarrow(cache_path)

    rows = []
    for var_name, code in indicators.items():
        url = f"https://api.worldbank.org/v2/country/all/indicator/{code}?format=json&per_page=20000&date={start}:{end}"
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        if not isinstance(data, list) or len(data) < 2:
            raise ValueError(f"Unexpected response for {code}")
        for item in data[1]:
            iso3 = (item.get('countryiso3code') or '').upper().strip()
            if not iso3:
                continue
            try:
                year = int(item['date'])
            except Exception:
                continue
            val = item.get('value')
            rows.append({
                'iso3': iso3,
                'year': year,
                'var_name': var_name,
                'value': val,
                'source': 'wdi',
                'indicator_code': code,
            })
        print('Fetched', var_name, code, 'rows:', len(data[1]))

    df = pd.DataFrame(rows)
    df['iso3'] = df['iso3'].str.upper().str.strip()
    df = df[df['iso3'].str.len() == 3]
    df = df[~df['iso3'].isin(AGGREGATES)]

    write_parquet_pyarrow(df, cache_path)

    # Also write a dated raw file for auditability
    dated = PATHS['data_raw'] / 'macro' / 'wdi' / f"wdi_long_{datetime.now().strftime('%Y%m%d')}.parquet"
    if not dated.exists():
        write_parquet_pyarrow(df, dated)

    # Update registry
    update_registry({
        'dataset': 'wdi',
        'pull_date': datetime.now().isoformat(timespec='seconds'),
        'start_year': start,
        'end_year': end,
        'indicators': indicators,
        'cache_path': str(cache_path),
    })

    return df


In [4]:
# Build PWT from local CSVs

PWT_CSV_PATHS = [
    ROOT / "pwt70-74.csv",
    ROOT / "pwt75-89.csv",
    ROOT / "pwt90-04.csv",
    ROOT / "pwt2005-23.csv",
]


def _read_pwt_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {"iso3", "year"}
    if not required.issubset(df.columns):
        raise ValueError(f"Expected columns {required} in {path.name}, got: {list(df.columns)}")

    df["iso3"] = df["iso3"].astype(str).str.upper().str.strip()
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

    df = df[df["iso3"].str.len() == 3]
    df = df[df["year"].notna()]
    df["year"] = df["year"].astype(int)

    # Keep only numeric value columns (drop text columns like Country)
    value_cols = [c for c in df.columns if c not in ["iso3", "year"]]

    numeric_cols = []
    for c in value_cols:
        converted = pd.to_numeric(df[c], errors="coerce")
        if converted.notna().any():
            df[c] = converted
            numeric_cols.append(c)

    keep_cols = ["iso3", "year"] + numeric_cols
    return df[keep_cols]


def build_pwt_from_csvs(paths: list[Path] | None = None) -> pd.DataFrame:
    paths = paths or PWT_CSV_PATHS
    missing = [p for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Missing PWT CSV(s): {[str(p) for p in missing]}")

    frames = [_read_pwt_csv(p) for p in paths]
    df = pd.concat(frames, ignore_index=True)

    # Drop duplicated iso3-year rows by keeping the last (later files should overwrite earlier if overlap)
    df = df.sort_values(["iso3", "year"]).drop_duplicates(subset=["iso3", "year"], keep="last")

    rename = {c: f"pwt_{c}" for c in df.columns if c not in ["iso3", "year"]}
    df = df.rename(columns=rename)

    return df


In [5]:
# Build EFW from Excel

EFW_XLSX_PATH = ROOT / "efw.xlsx"
EFW_SHEET = "EFW Panel Dataset"


def build_efw_from_excel(path: Path = EFW_XLSX_PATH, sheet: str = EFW_SHEET) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"EFW Excel not found: {path}")

    try:
        df = pd.read_excel(path, sheet_name=sheet)
    except ImportError as e:
        raise ImportError("Reading .xlsx requires openpyxl. Install via `pip install openpyxl`.") from e

    iso_candidates = {"iso3", "iso", "iso_code", "isocode", "country_code", "countrycode", "iso3c"}
    year_candidates = {"year", "yr"}

    iso_col = next((c for c in df.columns if c.strip().lower() in iso_candidates), None)
    year_col = next((c for c in df.columns if c.strip().lower() in year_candidates), None)

    if iso_col is None or year_col is None:
        raise ValueError(f"Could not find iso3/year columns in EFW Excel. Columns: {list(df.columns)}")

    out = df.copy()
    out["iso3"] = out[iso_col].astype(str).str.upper().str.strip()
    out["year"] = pd.to_numeric(out[year_col], errors="coerce")

    out = out[out["iso3"].str.len() == 3]
    out = out[out["year"].notna()]
    out["year"] = out["year"].astype(int)

    # Exclude key columns from numeric candidates
    value_cols = [c for c in out.columns if c not in [iso_col, year_col, "iso3", "year"]]

    numeric_cols = []
    for c in value_cols:
        converted = pd.to_numeric(out[c], errors="coerce")
        if converted.notna().any():
            out[c] = converted
            numeric_cols.append(c)

    keep_cols = ["iso3", "year"] + numeric_cols
    out = out[keep_cols].copy()

    rename = {}
    for c in numeric_cols:
        if c.startswith("efw_"):
            rename[c] = c
        else:
            rename[c] = f"efw_{c}"
    out = out.rename(columns=rename)

    out = out.drop_duplicates(subset=["iso3", "year"], keep="last")
    return out


In [6]:
# Build macro panel

def build_macro_panel(
    wdi_long: pd.DataFrame,
    *,
    pwt_wide: pd.DataFrame | None = None,
    efw_wide: pd.DataFrame | None = None,
    merge_how: str = "left",
) -> pd.DataFrame:
    # Pivot to wide
    wide = wdi_long.pivot_table(index=["iso3","year"], columns="var_name", values="value", aggfunc="mean").reset_index()

    # Transformations (WDI only)
    transforms = {
        "gdp_pc_real": "log",
        "pop": "log",
        "exrate_lcu_per_usd": "log"
    }

    for var, t in transforms.items():
        if var in wide.columns:
            newcol = f"macro_{var}_{t}"
            wide[newcol] = np.log(wide[var].astype(float))

    # Rename non-transformed to macro_* prefix (WDI only)
    for var in [c for c in wide.columns if c not in ["iso3","year"] and not c.startswith("macro_")]:
        if var in transforms:
            continue
        wide = wide.rename(columns={var: f"macro_{var}"})

    panel = wide

    if pwt_wide is not None and not pwt_wide.empty:
        # Basic merge integrity checks
        if panel.duplicated(subset=["iso3", "year"]).any():
            raise ValueError("WDI panel has duplicated iso3-year rows")
        if pwt_wide.duplicated(subset=["iso3", "year"]).any():
            raise ValueError("PWT data has duplicated iso3-year rows")

        base_keys = panel[["iso3", "year"]]
        pwt_keys = pwt_wide[["iso3", "year"]]
        overlap = base_keys.merge(pwt_keys, on=["iso3", "year"], how="outer", indicator=True)
        counts = overlap["_merge"].value_counts().to_dict()
        print("Merge check (WDI vs PWT):", counts)

        panel = panel.merge(pwt_wide, on=["iso3","year"], how=merge_how, validate="one_to_one")

    if efw_wide is not None and not efw_wide.empty:
        if efw_wide.duplicated(subset=["iso3", "year"]).any():
            raise ValueError("EFW data has duplicated iso3-year rows")

        base_keys = panel[["iso3", "year"]]
        efw_keys = efw_wide[["iso3", "year"]]
        overlap = base_keys.merge(efw_keys, on=["iso3", "year"], how="outer", indicator=True)
        counts = overlap["_merge"].value_counts().to_dict()
        print("Merge check (Panel vs EFW):", counts)

        panel = panel.merge(efw_wide, on=["iso3", "year"], how=merge_how, validate="one_to_one")

    # Coverage reports (WDI)
    macro_vars = [c for c in panel.columns if c.startswith("macro_")]
    cov = pd.DataFrame({
        "var": macro_vars,
        "missing_share_overall": [panel[v].isna().mean() for v in macro_vars],
    })
    write_parquet_pyarrow(cov, PATHS["reports"] / "macro_coverage" / "coverage_overall.parquet")

    # Coverage reports (PWT)
    pwt_vars = [c for c in panel.columns if c.startswith("pwt_")]
    if pwt_vars:
        cov_pwt = pd.DataFrame({
            "var": pwt_vars,
            "missing_share_overall": [panel[v].isna().mean() for v in pwt_vars],
        })
        write_parquet_pyarrow(cov_pwt, PATHS["reports"] / "macro_coverage" / "coverage_pwt_overall.parquet")

    # Coverage reports (EFW)
    efw_vars = [c for c in panel.columns if c.startswith("efw_")]
    if efw_vars:
        cov_efw = pd.DataFrame({
            "var": efw_vars,
            "missing_share_overall": [panel[v].isna().mean() for v in efw_vars],
        })
        write_parquet_pyarrow(cov_efw, PATHS["reports"] / "macro_coverage" / "coverage_efw_overall.parquet")

    return panel

# Fetch WDI
wdi_long = fetch_wdi(INDICATORS, start=1970, end=2023)

# Build PWT from CSVs
pwt_panel = build_pwt_from_csvs()

# Build EFW from Excel
# Place efw.xlsx at repo root (or adjust EFW_XLSX_PATH above)
efw_panel = build_efw_from_excel()

macro_panel = build_macro_panel(wdi_long, pwt_wide=pwt_panel, efw_wide=efw_panel)

# Save macro panel
out_path = PATHS["data_processed"] / "macro_panel.parquet"
write_parquet_pyarrow(macro_panel, out_path)
print("wrote", out_path)


Using cache: /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/raw/macro/wdi/cache_56c32c96762e.parquet
Merge check (WDI vs PWT): {'both': 9241, 'left_only': 2477, 'right_only': 162}
Merge check (Panel vs EFW): {'left_only': 6798, 'both': 4920, 'right_only': 30}
wrote /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/processed/macro_panel.parquet
